# TabAnywhere Gemma 4 E4B Unsloth Fine-Tuning

This notebook trains the unified rewrite-based edit prediction model with Unsloth QLoRA. It can run SFT, optional DPO preference tuning, then export GGUF variants for the macOS app.

Recommended runtime: **Colab Pro L4 or A100**. A free T4 can be used for a smoke test with small data, but it may be slow or memory-constrained.

Before running, set `REPO_URL` below to a GitHub URL containing this repo. If the repo is private, create a Colab secret named `GITHUB_TOKEN` with read access.

In [ ]:
#@title 1. Check GPU
# Avoid importing torch before the install cell; Colab can keep the old
# module loaded after pip upgrades, which causes CUDA extension mismatches.
!nvidia-smi

In [ ]:
#@title 2. Clone repo
REPO_URL = "https://github.com/Subjective/autocomplete"  # e.g. "https://github.com/YOUR_ORG/autocomplete.git"
BRANCH = ""    # optional; leave empty for default branch

assert REPO_URL, 'Set REPO_URL before running this cell.'

import os
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

clone_url = REPO_URL
if token and REPO_URL.startswith('https://github.com/'):
    clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)

%cd /content
!rm -rf autocomplete
branch_arg = f"--branch {BRANCH}" if BRANCH else ""
!git clone {branch_arg} {clone_url} autocomplete
%cd /content/autocomplete

In [ ]:
#@title 3. Install training dependencies
!pip install -U pip
!pip uninstall -y unsloth unsloth_zoo
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir "git+https://github.com/unslothai/unsloth-zoo.git"
!pip install --upgrade --no-cache-dir datasets huggingface_hub pyyaml accelerate trl peft

# Runtime sanity check
import unsloth, torch
print('Unsloth imported')
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
#@title 4. Optional Hugging Face login
# If google/gemma-4-E4B-it is gated, create a Colab secret named HF_TOKEN.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('Logged in to Hugging Face')
else:
    print('No HF_TOKEN secret found; continuing unauthenticated.')

In [ ]:
#@title 5. Configure workflow
RUN_SFT = True
RUN_DPO = True
EXPORT_GGUF = True

SFT_CONFIG = "training/configs/sft_gemma_lora.yaml"
DPO_CONFIG = "training/configs/dpo_gemma_lora.yaml"
SFT_EXPORT_CONFIG = "training/configs/export_gguf.yaml"
DPO_EXPORT_CONFIG = "training/configs/export_dpo_gguf.yaml"

SFT_MERGED_MODEL = "training/models/tabanywhere-gemma-edit-sft-v001-merged"
DPO_MERGED_MODEL = "training/models/tabanywhere-gemma-edit-dpo-v001-merged"
FINAL_MERGED_MODEL = DPO_MERGED_MODEL if RUN_DPO else SFT_MERGED_MODEL
EXPORT_CONFIG = DPO_EXPORT_CONFIG if RUN_DPO else SFT_EXPORT_CONFIG

print('RUN_SFT:', RUN_SFT)
print('RUN_DPO:', RUN_DPO)
print('EXPORT_GGUF:', EXPORT_GGUF)
print('Final export source:', FINAL_MERGED_MODEL)
print('Export config:', EXPORT_CONFIG)

In [ ]:
#@title 6. Validate and prepare data
!python3 training/scripts/validate_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl --kind sft
!python3 training/scripts/validate_dataset.py training/data/eval/tabanywhere_eval_v001.jsonl --kind eval
!python3 training/scripts/validate_dataset.py training/data/preference/tabanywhere_dpo_v001.jsonl --kind preference
!python3 training/scripts/split_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl --out-dir training/runs/splits
!python3 training/scripts/split_dataset.py training/data/preference/tabanywhere_dpo_v001.jsonl --out-dir training/runs/dpo_splits
!python3 training/scripts/prepare_sft_dataset.py training/data/seed/tabanywhere_seed_v001.jsonl training/runs/seed_messages.jsonl
!python3 training/scripts/prepare_dpo_dataset.py training/data/preference/tabanywhere_dpo_v001.jsonl training/runs/dpo_messages.jsonl

In [ ]:
#@title 7. Train SFT LoRA and merge
if RUN_SFT:
    !python3 training/scripts/train_unsloth_qlora.py --config {SFT_CONFIG} --merge
else:
    print('Skipping SFT stage')

In [ ]:
#@title 8. Optional DPO preference tuning and merge
if RUN_DPO:
    !python3 training/scripts/train_unsloth_dpo.py --config {DPO_CONFIG} --merge
else:
    print('Skipping DPO stage')

In [ ]:
#@title 9. Export GGUF variants
from pathlib import Path

if EXPORT_GGUF:
    if not Path(FINAL_MERGED_MODEL).exists():
        raise FileNotFoundError(f'Merged model not found: {FINAL_MERGED_MODEL}. Run the enabled training stages first.')
    !python3 training/scripts/export_gguf_unsloth.py --config {EXPORT_CONFIG} --source-model {FINAL_MERGED_MODEL}
    !find training/models -maxdepth 3 -type f | sort
else:
    print('Skipping GGUF export')

In [ ]:
#@title 10. Save artifacts to Google Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/tabanywhere-models
!cp -R training/models/* /content/drive/MyDrive/tabanywhere-models/
!find /content/drive/MyDrive/tabanywhere-models -maxdepth 3 -type f | sort